# 📊 Dashboard Interativo com Gradio para Detecção de Sessões Anômalas

In [1]:

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler
import sqlite3

# Conexão com banco de dados
con = sqlite3.connect("../database.db")
df = pd.read_sql_query("SELECT * FROM logs", con)

# Converter para datetime
df["data_hora"] = pd.to_datetime(df["data_hora"])

# Criar colunas numéricas derivadas
df["session_duration"] = df.groupby("usuario")["data_hora"].transform(lambda x: (x.max() - x.min()).total_seconds())
df["pages_visited"] = df.groupby("usuario")["acao"].transform("count")
df["session_duration"] = df["session_duration"].fillna(0)
df["pages_visited"] = df["pages_visited"].fillna(1)

# Normalização
scaler = MinMaxScaler()
x = scaler.fit_transform(df[["session_duration", "pages_visited"]])

# Isolation Forest
model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
df["anomalia"] = model.fit_predict(x)
df["anomalia"] = df["anomalia"].map({1: 0, -1: 1})
df["classificacao"] = df["anomalia"].map({0: "Normal", 1: "Suspeito"})


In [2]:

import gradio as gr

def filtrar(usuario):
    if usuario:
        filtrado = df[df["usuario"].str.contains(usuario, case=False, na=False)]
    else:
        filtrado = df.copy()
    return filtrado[["usuario", "acao", "session_duration", "pages_visited", "classificacao"]]

with gr.Blocks() as dashboard:
    gr.Markdown("**ℹ️ Nota:** A coluna `id` exibida representa o identificador do registro de log (evento), e não o identificador do usuário.")
    gr.Markdown("## 🔎 Filtrar Sessões por Usuário")
    usuario_input = gr.Textbox(label="Digite o nome ou parte do nome do usuário")
    btn = gr.Button("Filtrar")
    saida = gr.Dataframe()
    btn.click(fn=filtrar, inputs=usuario_input, outputs=saida)

dashboard.launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
